In [1]:
from datasets import load_dataset

ds1 = load_dataset('parquet', data_files='deepscaler/train.parquet', split='train')
ds2 = load_dataset('parquet', data_files='simplelr_math_35/train.parquet', split='train')

print(len(ds1))
print(len(ds2))


/home/chenluy/anaconda3/envs/new_verl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


40315
8523


In [7]:
ds1[0]

{'data_source': 'deepscaler',
 'prompt': [{'content': 'The operation $\\otimes$ is defined for all nonzero numbers by $a \\otimes b = \\frac{a^{2}}{b}$. Determine $[(1 \\otimes 2) \\otimes 3] - [1 \\otimes (2 \\otimes 3)]$.',
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': '-\\frac{2}{3}', 'style': 'rule'},
 'extra_info': {'index': 0, 'split': 'train'}}

In [5]:
ds2[0]

{'input': '<|im_start|>system\nPlease reason step by step, and put your final answer within \\boxed{}.<|im_end|>\n<|im_start|>user\nLet $a$ and $b$ be the two real values of $x$ for which\\[\\sqrt[3]{x} + \\sqrt[3]{20 - x} = 2\\]The smaller of the two values can be expressed as $p - \\sqrt{q}$, where $p$ and $q$ are integers. Compute $p + q$.<|im_end|>\n<|im_start|>assistant',
 'gt_answer': '118',
 'subject': 'Intermediate Algebra',
 'ground_truth_answer': '118',
 'target': '118',
 'data_source': 'simplelr_math_35',
 'prompt': [{'content': 'Let $a$ and $b$ be the two real values of $x$ for which\\[\\sqrt[3]{x} + \\sqrt[3]{20 - x} = 2\\]The smaller of the two values can be expressed as $p - \\sqrt{q}$, where $p$ and $q$ are integers. Compute $p + q$.',
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': '118', 'style': 'rule'},
 'extra_info': {'answer': '118',
  'index': 0,
  'level': 5,
  'question': 'Let $a$ and $b$ be the two real values of $x$ for which\\[\\sq

In [2]:
p = 0.2
n1 = int(len(ds1) * p)
n2 = int(len(ds2) * p)
print(n1, n2)

ds1_sft = ds1.shuffle(seed=42).select(range(n1))
ds1_remaining = ds1.shuffle(seed=42).select(range(n1, len(ds1)))

ds2_sft = ds2.shuffle(seed=42).select(range(n2))
ds2_remaining = ds2.shuffle(seed=42).select(range(n2, len(ds2)))


8063 1704


In [3]:
# 检查 ds1_sft 的结构
print("ds1_sft[0]:")
print(ds1_sft[0])
print("\nds1_sft extra_info 结构:")
print(ds1_sft[0]['extra_info'])
print("\nds1_sft extra_info 的字段:", list(ds1_sft[0]['extra_info'].keys()) if isinstance(ds1_sft[0]['extra_info'], dict) else "Not a dict")

ds1_sft[0]:
{'data_source': 'deepscaler', 'prompt': [{'content': 'Automobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeated exactly once, but digits cannot be repeated? [asy]\nsize(150);\ndraw((0,0)--(0,5)--(10,5)--(10,0)--cycle);\nlabel("\\Huge{CHIC - 03}",(1,3)--(9,3),S);\nlabel("\\small\\emph{State of Excellence}",(1,1)--(9,1),S);\ndraw((0.5,3.5)--(0.5,4.5)--(2,4.5)--(2,3.5)--cycle);\nlabel("\\footnotesize 5-03",(1.25,4));\ndraw((9.5,3.5)--(9.5,4.5)--(8,4.5)--(8,3.5)--cycle);\nlabel("\\footnotesize FX",(8.75,4));\n[/asy]', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '8,\\!424,\\!000', 'style': 'rule'}, 'extra_info': {'index': 35379, 'split': 'train'}}

ds1_sft extra_info 结构:
{'index': 35379, 'split': 'train'}

ds1_sft extra_info 的字段: ['index', 'split']


In [4]:
# 检查 ds2_sft 的原始结构
print("ds2_sft[0] (原始):")
print(ds2_sft[0])
print("\nds2_sft extra_info 结构:")
print(ds2_sft[0]['extra_info'])
print("\nds2_sft extra_info 的字段:", list(ds2_sft[0]['extra_info'].keys()) if isinstance(ds2_sft[0]['extra_info'], dict) else "Not a dict")

ds2_sft[0] (原始):
{'input': '<|im_start|>system\nPlease reason step by step, and put your final answer within \\boxed{}.<|im_end|>\n<|im_start|>user\n$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.<|im_end|>\n<|im_start|>assistant', 'gt_answer': '843', 'subject': 'Intermediate Algebra', 'ground_truth_answer': '843', 'target': '843', 'data_source': 'simplelr_math_35', 'prompt': [{'content': '$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '843', 'style': 'rule'}, 'extra_info': {'answer': '843', 'index': 6693, 'level': 5, 'question': '$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.', 'split': 'train'}}

ds2_sft extra_info 结构:
{'answer': '843', 'index': 6693, 'level': 5, 'question': 

In [5]:
# 先检查 ds1_sft 的 extra_info 结构，确保完全匹配
print("ds1_sft[0]['extra_info']:", ds1_sft[0]['extra_info'])
print("ds1_sft extra_info 字段:", list(ds1_sft[0]['extra_info'].keys()))

# 方法：先移除 extra_info，然后重新添加，确保结构完全匹配
from datasets import Features, Value

# 定义新的 extra_info 结构（只包含 index 和 split）
new_extra_info_features = {
    'index': Value('int64'),
    'split': Value('string')
}

def reorg(example):
    # 只保留需要的字段，并创建新的 extra_info
    return {
        'data_source': example['data_source'],
        'prompt': example['prompt'],
        'ability': example['ability'],
        'reward_model': example['reward_model'],
        # 创建新的 extra_info，只包含 index 和 split
        'new_extra_info': {
            'index': example['extra_info']['index'],
            'split': 'train'
        }
    }

# 先重组数据，将 extra_info 重命名为 new_extra_info
ds2_sft = ds2_sft.map(reorg, remove_columns=ds2_sft.column_names)

# 移除旧的 extra_info（如果还存在），并将 new_extra_info 重命名为 extra_info
def rename_extra_info(example):
    return {
        'extra_info': example['new_extra_info']
    }

ds2_sft = ds2_sft.map(rename_extra_info, remove_columns=['new_extra_info'])

# 现在明确指定 extra_info 的 features
from datasets import Features
target_extra_info_features = Features({
    'index': Value('int64'),
    'split': Value('string')
})

# 使用 cast_column 来明确转换 extra_info 的结构
ds2_sft = ds2_sft.cast_column('extra_info', target_extra_info_features)

print("\n重组后的 ds2_sft[0]:")
print(ds2_sft[0])
print("\n重组后的 ds2_sft extra_info:", ds2_sft[0]['extra_info'])

# 验证两个数据集的 features 是否一致
print("\n检查 features 是否一致:")
print("ds1_sft extra_info features:", ds1_sft.features['extra_info'])
print("ds2_sft extra_info features:", ds2_sft.features['extra_info'])
print("\n是否一致:", ds1_sft.features == ds2_sft.features)


ds1_sft[0]['extra_info']: {'index': 35379, 'split': 'train'}
ds1_sft extra_info 字段: ['index', 'split']


Map: 100%|██████████| 1704/1704 [00:00<00:00, 17126.99 examples/s]


重组后的 ds2_sft[0]:
{'data_source': 'simplelr_math_35', 'prompt': [{'content': '$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '843', 'style': 'rule'}, 'extra_info': {'index': 6693, 'split': 'train'}}

重组后的 ds2_sft extra_info: {'index': 6693, 'split': 'train'}

检查 features 是否一致:
ds1_sft extra_info features: {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}
ds2_sft extra_info features: {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}

是否一致: True


In [31]:
# 合并前最后检查 features 是否一致
print("合并前检查:")
print("ds1_sft features:", ds1_sft.features)
print("ds2_sft features:", ds2_sft.features)
print("\n是否一致:", ds1_sft.features == ds2_sft.features)

# 如果仍然不一致，使用 cast 统一整个 features
if ds1_sft.features != ds2_sft.features:
    print("\nFeatures 仍然不一致，使用 cast 统一整个 features...")
    ds2_sft = ds2_sft.cast(ds1_sft.features)
    print("转换后的 ds2_sft features:", ds2_sft.features)

# 现在可以合并了
from datasets import concatenate_datasets
ds_sft = concatenate_datasets([ds1_sft, ds2_sft])
print(f"\n✅ 合并成功！合并后的数据集大小: {len(ds_sft)}")
print(f"合并后的 features: {ds_sft.features}")

合并前检查:
ds1_sft features: {'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dtype='string', id=None)}], 'ability': Value(dtype='string', id=None), 'reward_model': {'ground_truth': Value(dtype='string', id=None), 'style': Value(dtype='string', id=None)}, 'extra_info': {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}}
ds2_sft features: {'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dtype='string', id=None)}], 'ability': Value(dtype='string', id=None), 'reward_model': {'ground_truth': Value(dtype='string', id=None), 'style': Value(dtype='string', id=None)}, 'extra_info': {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}}

是否一致: True

✅ 合并成功！合并后的数据集大小: 9767
合并后的 features: {'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dtype

In [32]:
len(ds_sft)

9767

In [33]:

ds_sft[0]

{'data_source': 'deepscaler',
 'prompt': [{'content': 'Automobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeated exactly once, but digits cannot be repeated? [asy]\nsize(150);\ndraw((0,0)--(0,5)--(10,5)--(10,0)--cycle);\nlabel("\\Huge{CHIC - 03}",(1,3)--(9,3),S);\nlabel("\\small\\emph{State of Excellence}",(1,1)--(9,1),S);\ndraw((0.5,3.5)--(0.5,4.5)--(2,4.5)--(2,3.5)--cycle);\nlabel("\\footnotesize 5-03",(1.25,4));\ndraw((9.5,3.5)--(9.5,4.5)--(8,4.5)--(8,3.5)--cycle);\nlabel("\\footnotesize FX",(8.75,4));\n[/asy]',
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': '8,\\!424,\\!000', 'style': 'rule'},
 'extra_info': {'index': 35379, 'split': 'train'}}

In [34]:
ds_sft.to_parquet('sft.parquet')

Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 35.80ba/s]


3097601

In [6]:
ds1_remaining.to_parquet('deepscaler/train_remaining.parquet')
ds2_remaining.to_parquet('simplelr_math_35/train_remaining.parquet')

Creating parquet from Arrow format: 100%|██████████| 7/7 [00:00<00:00, 19.03ba/s]


6605456

In [1]:
from datasets import load_dataset
ds0 = load_dataset('json', data_files='/home/chenluy/SimpleTIR/sft/sft_data.jsonl', split='train')
ds0[0]

/home/chenluy/anaconda3/envs/new_verl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 1188842 examples [00:06, 180301.61 examples/s]


{'messages': [{'content': 'If $G(m, n, p, q) = m^n + p \\times q$, what is the value of $y$ such that $G(3, y, 6, 15) = 171$?',
   'role': 'user'},
  {'content': '```python\n# Step 1: Define the function G(m, n, p, q)\ndef G(m, n, p, q):\n    return m**n + p * q\n\n# Step 2: Set up the equation G(3, y, 6, 15) = 171\nequation = G(3, y, 6, 15) - 171\n\n# Step 3: Solve for y\n# We need to find y such that 3^y + 6 * 15 = 171\n# Simplify the equation: 3^y + 90 = 171\n# Therefore, 3^y = 81\n# We know that 81 = 3^4, so y = 4\n\n# Now print the final answer\nprint(4)\n```\nCode execution result: 4\nThe value of y is \\boxed{4}.',
   'role': 'assistant'}]}

In [3]:
ds0[3]

{'messages': [{'content': 'Every evening when Sarah gets ready for bed, she randomly chooses a pair of slippers from her closet without looking. If she has 6 pink slippers and 4 blue slippers, what is the probability that she picks a matching pair of slippers?',
   'role': 'user'},
  {'content': '```python\n# Step 1: Calculate the total number of ways to choose 2 slippers out of 10\nfrom math import comb\n\ntotal_ways = comb(10, 2)\n\n# Step 2: Calculate the number of ways to choose 2 pink slippers\npink_ways = comb(6, 2)\n\n# Step 3: Calculate the number of ways to choose 2 blue slippers\nblue_ways = comb(4, 2)\n\n\n# Step 4: Calculate the probability of picking a matching pair of slippers\nprobability = (pink_ways + blue_ways) / total_ways\n\n# Now print the final answer\nprint(probability)\n```\nCode execution result: 0.4667\nThe probability that Sarah picks a matching pair of slippers is \\boxed{0.4667}.',
   'role': 'assistant'}]}

## Deal with rstar2

In [30]:
ds = load_dataset('parquet', data_files='/home/chenluy/SimpleTIR/datasets/rstar2_dataset/train.parquet', split='train')
ds[0]

{'data_source': 'custom_math_DAPO-Math-17k-Processed',
 'prompt': [{'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': '34', 'style': 'rule'},
 'extra_info': {'index': 0, 'split': 'train'}}

In [ ]:
import re

def change_format(example):
    # Extract the question from the 'input' field
    input_text = example['input']
    
    # Use regex to extract text between <|im_start|>user\n and <|im_end|>
    match = re.search(r'<\|im_start\|>user\n(.*?)<\|im_end\|>', input_text, re.DOTALL)
    
    if match:
        question = match.group(1).strip()
    else:
        # Fallback: if pattern not found, try to use existing prompt or empty string
        question = example.get('prompt', [{}])[0].get('content', '') if isinstance(example.get('prompt'), list) else ''
    
    # Get ground_truth from available fields
    ground_truth = example.get('gt_answer') or example.get('ground_truth_answer') or example.get('target', '')
    
    # Get index and split from extra_info
    orig_extra_info = example.get('extra_info', {})
    index_val = orig_extra_info.get('index', 0) if isinstance(orig_extra_info, dict) else 0
    split_val = orig_extra_info.get('split', 'train') if isinstance(orig_extra_info, dict) else 'train'
    
    # Return only the fields needed to match the target format
    return {
        'data_source': example['data_source'],
        'prompt': [{'content': question, 'role': 'user'}],
        'ability': example['ability'],
        'reward_model': {
            'ground_truth': str(ground_truth),
            'style': 'rule'
        },
        'extra_info': {
            'index': index_val,
            'split': split_val
        }
    }
    

In [32]:
# Test the function on the first example
print("Before transformation:")
print(ds[0])
print("\n" + "="*80 + "\n")

# Apply the transformation
test_result = change_format(ds[0])
print("After transformation:")
print(test_result)
print("\nKeys:", list(test_result.keys()))
print("Extra info keys:", list(test_result['extra_info'].keys()))


Before transformation:
{'data_source': 'custom_math_DAPO-Math-17k-Processed', 'prompt': [{'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '34', 'style': 'rule'}, 'extra_info': {'index': 0, 'split': 'train'}}




KeyError: 'input'

In [24]:
# Apply transformation - first to Python dicts, then create new dataset with correct features
from datasets import Dataset, Features, Value

print("Transforming dataset to Python dicts...")
# Transform all examples to Python dicts
transformed_data = []
for i in range(len(ds)):
    transformed_data.append(change_format(ds[i]))
    if (i + 1) % 10000 == 0:
        print(f"Processed {i + 1}/{len(ds)} examples...")

print(f"\n✅ Transformed {len(transformed_data)} examples")
print(f"\nSample transformed example:")
print(transformed_data[0])

# Get the target features from ds1
print("\n" + "="*80)
print("Target features (from ds1):")
print(f"{ds1.features}")

# Create new dataset from the transformed data with explicit features
print("\n" + "="*80)
print("Creating new dataset with target features...")
ds_transformed = Dataset.from_dict({
    'data_source': [d['data_source'] for d in transformed_data],
    'prompt': [d['prompt'] for d in transformed_data],
    'ability': [d['ability'] for d in transformed_data],
    'reward_model': [d['reward_model'] for d in transformed_data],
    'extra_info': [d['extra_info'] for d in transformed_data]
}, features=ds1.features)

print(f"\n✅ Dataset created with correct features!")
print(f"Transformed dataset features:\n{ds_transformed.features}")
print(f"\nFirst example:\n{ds_transformed[0]}")
print(f"\nFeatures match: {ds_transformed.features == ds1.features}")




Transforming dataset to Python dicts...
Processed 10000/42266 examples...
Processed 20000/42266 examples...
Processed 30000/42266 examples...
Processed 40000/42266 examples...

✅ Transformed 42266 examples

Sample transformed example:
{'data_source': 'custom_math_DAPO-Math-17k-Processed', 'prompt': [{'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '34', 'style': 'rule'}, 'extra_info': {'index': 0, 'split': 'train'}}

Target features (from ds1):
{'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dt

In [25]:
# This cell is now optional - casting is done in cell 18
# You can run this cell for additional verification if needed
print("Final verification:")
print(f"Dataset size: {len(ds_transformed)}")
print(f"Features: {ds_transformed.features}")
print(f"Sample: {ds_transformed[0]}")


Final verification:
Dataset size: 42266
Features: {'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dtype='string', id=None)}], 'ability': Value(dtype='string', id=None), 'reward_model': {'ground_truth': Value(dtype='string', id=None), 'style': Value(dtype='string', id=None)}, 'extra_info': {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}}
Sample: {'data_source': 'custom_math_DAPO-Math-17k-Processed', 'prompt': [{'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '34

In [26]:
# Save the transformed dataset
output_path = '/home/chenluy/SimpleTIR/datasets/rstar2_dataset/train.parquet'
ds_transformed.to_parquet(output_path)
print(f"✅ Transformed dataset saved to: {output_path}")


Creating parquet from Arrow format: 100%|██████████| 43/43 [00:00<00:00, 733.24ba/s]

✅ Transformed dataset saved to: /home/chenluy/SimpleTIR/datasets/rstar2_dataset/train.parquet
